# ML-07 — Baseline Action Score and Top-20 Review

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [2]:
df['staleness_bucket'] = pd.cut(
    df['days_since_last_update'],
    bins=[0, 90, 180, 365, np.inf],
    labels=['<90d', '90-180d', '180-365d', '365d+']
)

staleness_table = df.groupby('staleness_bucket', observed=True).agg(
    n=('content_id', 'count'),
    declining_rate=('trend_direction', lambda x: (x.str.lower() == 'down').mean())
).round(3)

print(staleness_table)

                      n  declining_rate
staleness_bucket                       
<90d              20655           0.512
90-180d            9171           0.611
180-365d            169           0.467
365d+                 5           0.600


In [4]:
visible = df[(df['avg_position'] > 0) & (df['impressions_90d'] >= 100)].copy()
visible['position_bucket'] = pd.cut(
    visible['avg_position'],
    bins=[0, 3, 10, 20, np.inf],
    labels=['1-3', '4-10', '11-20', '20+']
)

ctr_table = visible.groupby('position_bucket', observed=True).agg(
    n=('content_id', 'count'),
    mean_ctr=('ctr', 'mean')
).round(3)

print(ctr_table)


                    n  mean_ctr
position_bucket                
1-3               555     0.337
4-10             8660     0.354
11-20            5876     0.256
20+              6915     0.131


The declining rate rises from <90d (0.512) to 90-180d (0.611), which fits the expected staleness story, but then drops to 0.467 in the 180-365d bucket, breaking the pattern. The 365d+ bucket shows a higher rate again (0.600), but with only n=5 rows, that number is unreliable and shouldn't be trusted as a real signal. 

CTR drops sharply and consistently from position 4-10 (0.354) through 11-20 (0.256) to 20+ (0.131). This strongly confirms the core CTR-vs-position relationship the CTR-fix flag logic relies on. However, the top bucket (position 1-3, CTR 0.337) is unexpectedly lower than the 4-10 bucket (0.354), which breaks the naive assumption that CTR always increases monotonically as position improves.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [5]:
 

df['stale_flag'] = (df['days_since_last_update'] >= 180).astype(int)

df['visible_flag'] = (df['impressions_90d'] >= 500).astype(int)

df['baseline_action_score'] = (

    df['stale_flag'] * 0.6 * np.log1p(df['impressions_90d']) +

    df['visible_flag'] * 0.4 * np.log1p(df['impressions_90d'])

)



df['reason_code'] = np.where(

    (df['stale_flag'] == 1) & (df['visible_flag'] == 1),

    'stale_visible_page',

    'other'

)



df['action'] = np.where(df['reason_code'] == 'stale_visible_page', 'review_for_refresh', 'monitor')

queue = df.sort_values('baseline_action_score', ascending=False)

os.makedirs('work/outputs', exist_ok=True)

queue[['content_id', 'client_id', 'baseline_action_score', 'reason_code', 'action',

       'days_since_last_update', 'impressions_90d', 'avg_position', 'ctr', 'trend_direction']].to_csv(

    'work/outputs/baseline_action_score.csv', index=False

)

print(f"Wrote {len(queue):,} rows to work/outputs/baseline_action_score.csv")

queue[['content_id', 'baseline_action_score', 'reason_code', 'action']].head(10)

Wrote 30,000 rows to work/outputs/baseline_action_score.csv


,content_id,baseline_action_score,reason_code,action
16751,content_cf56e2e2e282,11.029699,stale_visible_page,review_for_refresh
16514,content_7368877ea310,10.993278,stale_visible_page,review_for_refresh
7021,content_1bfaa38ff26c,10.154869,stale_visible_page,review_for_refresh
21268,content_0a91db491d14,9.495519,stale_visible_page,review_for_refresh
11489,content_5feee3994adb,8.963544,stale_visible_page,review_for_refresh
12045,content_c2d929d83eaa,8.930494,stale_visible_page,review_for_refresh
698,content_b16bd7307b39,8.431853,stale_visible_page,review_for_refresh
5327,content_fe16a55cd13d,8.424420,stale_visible_page,review_for_refresh
26810,content_ecb6215e79fd,8.396155,stale_visible_page,review_for_refresh
20837,content_928af3e22c80,7.437206,stale_visible_page,review_for_refresh


In [6]:
top20 = queue.head(20)[['content_id', 'action', 'reason_code', 'baseline_action_score',
                          'days_since_last_update', 'impressions_90d', 'avg_position',
                          'ctr', 'engagement_rate', 'trend_direction']]
top20

,content_id,action,reason_code,baseline_action_score,days_since_last_update,impressions_90d,avg_position,ctr,engagement_rate,trend_direction
16751,content_cf56e2e2e282,review_for_refresh,stale_visible_page,11.029699,194,61678,19.7,0.15,0.84,down
16514,content_7368877ea310,review_for_refresh,stale_visible_page,10.993278,194,59472,24.8,0.13,3.66,down
7021,content_1bfaa38ff26c,review_for_refresh,stale_visible_page,10.154869,194,25715,22.2,0.23,3.75,down
21268,content_0a91db491d14,review_for_refresh,stale_visible_page,9.495519,193,13299,10.5,0.49,5.13,down
11489,content_5feee3994adb,review_for_refresh,stale_visible_page,8.963544,194,7812,39.0,0.01,0.00,down
12045,content_c2d929d83eaa,review_for_refresh,stale_visible_page,8.930494,193,7558,17.9,0.20,0.00,down
698,content_b16bd7307b39,review_for_refresh,stale_visible_page,8.431853,194,4590,31.0,0.00,0.00,down
5327,content_fe16a55cd13d,review_for_refresh,stale_visible_page,8.424420,194,4556,16.4,0.33,2.38,down
26810,content_ecb6215e79fd,review_for_refresh,stale_visible_page,8.396155,194,4429,25.3,0.38,25.00,down
20837,content_928af3e22c80,review_for_refresh,stale_visible_page,7.437206,193,1697,15.8,0.12,0.00,down


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

content_cf56e2e2e282 — review_for_refresh. Stale (194d), massive visibility (61,678 impressions), declining, weak CTR (0.15) for its position. Confidence: high. Wrong if: this traffic is seasonal/event-driven and naturally tapering, not a content quality issue.
content_7368877ea310 — review_for_refresh. Similar profile, stale, high traffic, declining, low CTR. Confidence: high. Wrong if: same seasonal risk as above.
content_1bfaa38ff26c — review_for_refresh. Stale, solid traffic (25,715), declining. Confidence: medium. Wrong if: position (22.2) is already weak enough that a refresh won't move the needle without a bigger SEO fix.
content_0a91db491d14 — review_for_refresh. Stale, decent CTR (0.49) and okay position (10.5) already. Confidence: low-medium. Wrong if: this page is already performing reasonably (0.49 CTR is strong) — refresh may not be the right action; something else (e.g. expansion) might fit better.
content_5feee3994adb — review_for_refresh. Stale, weak position (39.0), near-zero CTR (0.01) and zero engagement. Confidence: high. Wrong if: this page never had real intent-match to begin with — a refresh won't fix a fundamentally mismatched page.
content_c2d929d83eaa — review_for_refresh. Stale, zero engagement, moderate traffic. Confidence: medium. Wrong if: engagement_rate = 0.00 reflects a tracking gap (GA4 not available) rather than real zero engagement — worth checking ga4_data_available before trusting this number.
content_b16bd7307b39 — review_for_refresh. Stale, zero CTR, zero engagement, weak position (31.0). Confidence: high. Wrong if: same GA4-availability caveat as #6.
content_fe16a55cd13d — review_for_refresh. Stale, moderate CTR (0.33), some engagement. Confidence: medium. Wrong if: this is a slow-and-steady page where "stale" doesn't equal "broken."
content_ecb6215e79fd — review_for_refresh. Stale, decent CTR (0.38), and unusually high engagement (25.00). Confidence: low. Likely a weak pick — high engagement contradicts the "needs refresh" story; the rule flagged it purely on staleness + visibility, missing that engagement is actually strong.
content_928af3e22c80 — review_for_refresh. Stale, lower traffic (1,697), weak CTR. Confidence: medium. Wrong if: low absolute traffic means this page isn't worth reviewer time regardless of the score.
content_e3ff1b093148 — review_for_refresh. Stale (183d), decent position (7.8) already, moderate CTR. Confidence: low. Wrong if: position is already strong — this reads more like a stable performer than a real refresh candidate.
content_bdbec75c1148 — review_for_refresh, but trend_direction = stable, not down. Confidence: low. This is a genuinely interesting weak pick — the rule flagged it purely on staleness/visibility, and it turns out to be stable, not declining, showing the rule can't distinguish "stale but fine" from "stale and actually failing."
content_7f116ae1f6f5 — review_for_refresh. Very stale (301d), lower traffic (954), zero engagement. Confidence: medium. Wrong if: low traffic means limited upside even if genuinely declining.
content_77d4d5930e5e — review_for_refresh. Stale, but engagement_rate = 50.00 — extremely high, likely an outlier or scaling artifact. Confidence: low. Weak pick — this number looks suspicious (way outside the normal range seen elsewhere) and should be sanity-checked before trusting the flag.
content_72496874f806 — review_for_refresh. Very stale (301d), low traffic, decent position (5.8) already. Confidence: low. Wrong if: strong position despite staleness suggests this page is doing fine without intervention.
content_6226ee6adc91 — review_for_refresh. Stale, low traffic, weak CTR. Confidence: medium. Wrong if: traffic is too low to justify reviewer time.
content_074ba6ead17b — review_for_refresh. Stale, very weak position (48.0), zero CTR/engagement. Confidence: high. This looks like a genuinely dead page — strong candidate.
content_5fe46e04994d — monitor. Not stale (104d), but huge traffic (517,715). Confidence: n/a (correctly excluded from refresh). Right call — too recently updated to flag as stale despite scale.
content_aaef01a50def — monitor. Barely stale (22d), massive traffic (517,109), stable trend. Confidence: n/a. Correctly excluded — fresh and performing.
content_8c19996aa890 — monitor. Very fresh (20d), huge traffic (509,252). Confidence: n/a. Correctly excluded — same reasoning.

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

Weak picks identified: Row #12 (content_bdbec75c1148) is the clearest weak pick. It was selected for a refresh because it is old and has low visibility, but its actual trend_direction is stable, not declining. This shows a problem with the rule: being old does not always mean that a page is performing badly.
Rows #9 and #14 are also weak picks. Both have unusually high engagement_rate values (25.00 and 50.00), which does not match the idea that they need a refresh. The value for #14 especially looks like it could be an unusual value or a data scaling issue, so it should be checked against the data dictionary.
Leakage check: The rule only uses days_since_last_update and impressions_90d. These are real values that were available before the result was known. trend_direction and trend_pct were not used to create the score, reason code, or action. I only checked trend_direction afterward to see if the rule's choices made sense, such as with row #12. This is okay because it was used only for reviewing the results, not as an input to the rule.
No product-generated values such as health_score or priority_score were used because they are not included in this dataset. There were also no future results or label-based values used to create the score.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.